In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")


Using device: mps


In [ ]:
model_name = "cross-encoder/stsb-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()
print(f"Loaded model: {model_name}")
print(f"Number of labels: {model.config.num_labels}")


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])


In [ ]:
sentence1_lengths = []
sentence2_lengths = []
length_diffs = []
length_ratios = []

for row in dataset:
    enc1 = tokenizer(row["sentence1"], truncation=False, add_special_tokens=False)
    enc2 = tokenizer(row["sentence2"], truncation=False, add_special_tokens=False)
    len1 = len(enc1["input_ids"])
    len2 = len(enc2["input_ids"])
    diff = abs(len1 - len2)
    ratio = diff / max(len1, len2) if max(len1, len2) > 0 else 0.0
    sentence1_lengths.append(len1)
    sentence2_lengths.append(len2)
    length_diffs.append(diff)
    length_ratios.append(ratio)

sorted_ratios = sorted(length_ratios)
n = len(sorted_ratios)
low_threshold = sorted_ratios[n // 3]
medium_threshold = sorted_ratios[(2 * n) // 3]

def assign_asymmetry_bucket(ratio):
    if ratio <= low_threshold:
        return "low"
    elif ratio <= medium_threshold:
        return "medium"
    return "high"

asymmetry_buckets = [assign_asymmetry_bucket(r) for r in length_ratios]
bucket_counts = {name: asymmetry_buckets.count(name) for name in ["low", "medium", "high"]}

print(f"Asymmetry ratio thresholds -> low <= {low_threshold:.4f}, medium <= {medium_threshold:.4f}, high > {medium_threshold:.4f}")
print("Bucket counts:")
print(bucket_counts)
print(f"Sentence1 length min/max/avg: {min(sentence1_lengths)} / {max(sentence1_lengths)} / {sum(sentence1_lengths)/len(sentence1_lengths):.2f}")
print(f"Sentence2 length min/max/avg: {min(sentence2_lengths)} / {max(sentence2_lengths)} / {sum(sentence2_lengths)/len(sentence2_lengths):.2f}")
print(f"Asymmetry diff min/max/avg: {min(length_diffs)} / {max(length_diffs)} / {sum(length_diffs)/len(length_diffs):.2f}")
print(f"Asymmetry ratio min/max/avg: {min(length_ratios):.4f} / {max(length_ratios):.4f} / {sum(length_ratios)/len(length_ratios):.4f}")


In [ ]:
batch_size = 32
labels = dataset["label"]
predictions = []
confidences = []
raw_scores = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        if logits.shape[-1] == 1:
            scores = torch.sigmoid(logits.squeeze(-1))
            preds = (scores >= 0.5).long()
            conf = torch.where(preds == 1, scores, 1 - scores)
            raw = scores
        else:
            probs = torch.softmax(logits, dim=-1)
            preds = torch.argmax(probs, dim=-1)
            conf = probs.max(dim=-1).values
            raw = probs[:, 1] if probs.shape[-1] > 1 else conf
    predictions.extend(preds.cpu().tolist())
    confidences.extend(conf.cpu().tolist())
    raw_scores.extend(raw.cpu().tolist())

print(f"Completed inference for {len(predictions)} examples.")
print(f"Positive prediction rate: {sum(predictions)/len(predictions):.4f}")


In [ ]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, predictions)

print("Overall evaluation metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)

per_true_label_report = classification_report(
    labels,
    predictions,
    labels=[0, 1],
    target_names=["not_paraphrase", "paraphrase"],
    zero_division=0,
    output_dict=True
)

print("Per-true-label precision/recall/f1/support:")
for name in ["not_paraphrase", "paraphrase"]:
    m = per_true_label_report[name]
    print(f"label={name} precision={m['precision']:.4f} recall={m['recall']:.4f} f1={m['f1-score']:.4f} support={int(m['support'])}")


In [ ]:
bucket_metrics = {}

for bucket_name in ["low", "medium", "high"]:
    idxs = [i for i, b in enumerate(asymmetry_buckets) if b == bucket_name]
    y_true = [labels[i] for i in idxs]
    y_pred = [predictions[i] for i in idxs]
    bucket_accuracy = accuracy_score(y_true, y_pred)
    bucket_precision, bucket_recall, bucket_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    bucket_cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    avg_diff = sum(length_diffs[i] for i in idxs) / len(idxs)
    avg_ratio = sum(length_ratios[i] for i in idxs) / len(idxs)
    bucket_metrics[bucket_name] = {
        "count": len(idxs),
        "accuracy": bucket_accuracy,
        "precision": bucket_precision,
        "recall": bucket_recall,
        "f1": bucket_f1,
        "avg_length_diff": avg_diff,
        "avg_length_ratio": avg_ratio,
        "confusion_matrix": bucket_cm.tolist()
    }

print("Asymmetry bucket metrics:")
for bucket_name in ["low", "medium", "high"]:
    m = bucket_metrics[bucket_name]
    print(
        f"bucket={bucket_name} count={m['count']} accuracy={m['accuracy']:.4f} "
        f"precision={m['precision']:.4f} recall={m['recall']:.4f} f1={m['f1']:.4f} "
        f"avg_length_diff={m['avg_length_diff']:.2f} avg_length_ratio={m['avg_length_ratio']:.4f}"
    )
    print(f"confusion_matrix={m['confusion_matrix']}")


In [ ]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}
asymmetric_mistakes = []

for i, bucket_name in enumerate(asymmetry_buckets):
    if bucket_name == "high" and labels[i] != predictions[i]:
        asymmetric_mistakes.append({
            "idx": i,
            "len1": sentence1_lengths[i],
            "len2": sentence2_lengths[i],
            "length_diff": length_diffs[i],
            "length_ratio": length_ratios[i],
            "true_label": labels[i],
            "pred_label": predictions[i],
            "confidence": confidences[i],
            "score": raw_scores[i],
            "sentence1": dataset[i]["sentence1"],
            "sentence2": dataset[i]["sentence2"]
        })

asymmetric_mistakes = sorted(asymmetric_mistakes, key=lambda x: (-x["length_ratio"], -x["length_diff"], -x["confidence"]))
num_examples_to_show = min(5, len(asymmetric_mistakes))

print(f"Representative mistakes from high-asymmetry bucket: showing {num_examples_to_show} of {len(asymmetric_mistakes)}")
for item in asymmetric_mistakes[:num_examples_to_show]:
    print(f"Index: {item['idx']}")
    print(f"Bucket: high | sentence1_length: {item['len1']} | sentence2_length: {item['len2']} | length_diff: {item['length_diff']} | length_ratio: {item['length_ratio']:.4f}")
    print(f"sentence1: {item['sentence1']}")
    print(f"sentence2: {item['sentence2']}")
    print(f"true label: {item['true_label']} ({label_map[item['true_label']]})")
    print(f"pred label: {item['pred_label']} ({label_map[item['pred_label']]})")
    print(f"confidence: {item['confidence']:.4f} | positive_score: {item['score']:.4f}")
    print("-" * 80)


In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"overall_accuracy={accuracy:.4f}")
print(f"overall_precision={precision:.4f}")
print(f"overall_recall={recall:.4f}")
print(f"overall_f1={f1:.4f}")
print(f"asymmetry_threshold_low={low_threshold:.4f}")
print(f"asymmetry_threshold_medium={medium_threshold:.4f}")
for label_name in ["not_paraphrase", "paraphrase"]:
    m = per_true_label_report[label_name]
    key = label_name
    print(f"true_label_{key}_precision={m['precision']:.4f}")
    print(f"true_label_{key}_recall={m['recall']:.4f}")
    print(f"true_label_{key}_f1={m['f1-score']:.4f}")
    print(f"true_label_{key}_support={int(m['support'])}")
for bucket_name in ["low", "medium", "high"]:
    m = bucket_metrics[bucket_name]
    print(f"bucket_{bucket_name}_count={m['count']}")
    print(f"bucket_{bucket_name}_accuracy={m['accuracy']:.4f}")
    print(f"bucket_{bucket_name}_precision={m['precision']:.4f}")
    print(f"bucket_{bucket_name}_recall={m['recall']:.4f}")
    print(f"bucket_{bucket_name}_f1={m['f1']:.4f}")
    print(f"bucket_{bucket_name}_avg_length_diff={m['avg_length_diff']:.2f}")
    print(f"bucket_{bucket_name}_avg_length_ratio={m['avg_length_ratio']:.4f}")
